# 2 — Order parameter (−S_CD)

All-atom CHARMM36 binary bilayer (DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA).

Per-carbon deuterium order parameter for sn-1 (`C3x`) and sn-2 (`C2x`) chains; C–H bonds auto-detected. On a single frame this is the spatial average over all lipids (error = spread across molecules).

> Uses a `.tpr` so C–H bonds are known (needed here).


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config — edit these

You have **no trajectory**, so `TRAJ = None` loads the single `.gro`
frame (a **snapshot**). When you have an `.xtc`/`.trr`, set `TRAJ` to it
and every cell below averages over frames automatically.

In [ ]:
TOPOLOGY = "md.tpr"        # topology (has bonds + masses)
STRUCTURE= "md.gro"        # single-frame structure (.gro or .pdb)
TRAJ     = None            # e.g. "md_center.xtc" when you have one; None -> use STRUCTURE

LIPIDS   = ["DOPC", "DOPA"]        # the TWO lipids in THIS system
HEAD     = "P"                     # headgroup / leaflet marker
STEP     = 1                       # frame stride (only matters with a trajectory)
OUTDIR   = "results"; os.makedirs(OUTDIR, exist_ok=True)

## Load

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ) if TRAJ else mda.Universe(TOPOLOGY, STRUCTURE)
has_bonds = len(getattr(u, "bonds", [])) > 0
present   = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
nframes   = len(u.trajectory)
print(f"{len(u.atoms)} atoms, {nframes} frame(s), bonds={has_bonds}")
print("Lipids present:", present)
if nframes == 1:
    print("NOTE: single frame -> snapshot only (no time averaging).")

## Compute −S_CD

In [ ]:
def carbons(res, prefix):
    pat = re.compile(r"^"+prefix+r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m:=pat.match(at.name)) and int(m.group(1))>=2]
    return sorted(hits, key=lambda t:t[0])

def hydrogens(catom):
    if has_bonds: return [a for a in catom.bonded_atoms if a.name.startswith("H")]
    return list(u.select_atoms("name H* and around 1.3 group c", c=u.atoms[[catom.index]]))

order = {}
for res in present:
    for prefix, tag in (("C3","sn1"),("C2","sn2")):
        first = u.select_atoms(f"resname {res}").residues[0]
        cs = carbons(first, prefix)
        if not cs: continue
        cidx = [i for i,_ in cs]; acc = {i:[] for i in cidx}
        for ts in u.trajectory[::STEP]:
            for r in u.select_atoms(f"resname {res}").residues:
                for idx, catom in carbons(r, prefix):
                    hs = hydrogens(catom)
                    if not hs: continue
                    cp = catom.position
                    vv = [0.5*(3*((h.position-cp)/np.linalg.norm(h.position-cp))[2]**2 - 1) for h in hs]
                    acc[idx].append(np.mean(vv))
        i,s,e = [],[],[]
        for k in cidx:
            a=np.array(acc[k])
            if a.size: i.append(k); s.append(-a.mean()); e.append(a.std()/np.sqrt(a.size))
        order[(res,tag)] = (np.array(i),np.array(s),np.array(e))
        np.savetxt(f"{OUTDIR}/order_{res}_{tag}.dat", np.column_stack([i,s,e]), header="C -SCD sem")

plt.figure(figsize=(7,5))
for (res,tag),(i,s,e) in order.items():
    plt.errorbar(i,s,yerr=e,marker="o",capsize=2,label=f"{res} {tag}")
plt.xlabel("Carbon number"); plt.ylabel(r"$-S_{CD}$"); plt.legend(fontsize=8)
plt.grid(alpha=.3); plt.title("Order parameters"); plt.show()